In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from utils import eda_toolkit as toolkit
from utils import build_features as bf

train = pd.read_parquet(DATA_PROCESSED / "train.parquet")
holdout = pd.read_parquet(DATA_PROCESSED / "holdout.parquet")

# Ingeniería de variables

:::{admonition} Resumen de la ingeniería de variables
:class: info

- Se derivaron 22 variables predictoras, agrupadas en calendario, codificación cíclica, estacionalidad y rezagos.
- Los rezagos son 32, 33, 65 y 455. Los desplazamientos menores no están disponibles al momento de decidir.
- La codificación cíclica usa seno-coseno con armónicas adicionales en hora y mes. Así se captura la bimodalidad intradía y la asimetría anual.
- La respuesta `pasajeros` se entrega sin transformaciones. Los targets por horizonte (30, 60, 120 minutos) se construyen en la fase de modelado.
- El pipeline produce el mismo esquema en entrenamiento y validación, sin nulos en las predictoras tras el warmup, y aplica igual a cualquier split futuro.
:::

## Introducción

Este cuaderno documenta la fase de ingeniería de variables del proyecto *Data Challenge Pro 2026 — Optimización de frecuencias de buses SOBUSA*, desarrollado en conjunto por la Universidad del Norte, Sobusa S.A. y Extreme Technologies. Su propósito es transformar los consolidados operacionales de la ruta en un conjunto de variables analíticas reproducibles, listas para el entrenamiento y la inferencia de los modelos de predicción de demanda.

El cuaderno está diseñado como plantilla reutilizable para las tres rutas consideradas en el proyecto: **CRA 54**, **SILENCIO** y **CL 72**. La metodología de análisis exploratorio ya fue unificada entre rutas, de modo que las decisiones de limpieza, agregación y codificación aquí descritas aplican de forma idéntica a cada una. Las particularidades operativas de cada ruta quedan encapsuladas en su consolidado correspondiente y no alteran el pipeline descrito.

:::{admonition} Información: Dataset de Demanda — Rutas del Proyecto
:class: info

El consolidado operacional de cada ruta comprende una serie temporal agregada por franjas de 15 minutos, que cubre la ventana operativa de la ruta a lo largo de los años 2023–2025. Cada franja representa el total de pasajeros transportados por los viajes cuyo despacho ocurrió en ese intervalo. La partición del conjunto es cronológica, con 80 % asignado al entrenamiento y 20 % a la validación.

Cada registro contiene las siguientes columnas iniciales: `franja`, `anio`, `mes`, `dia`, `dia_semana`, `tipo_estacionalidad`,
  `nombre_evento`, `pasajeros`, `n_viajes`.

Sobre ellas se derivan las variables analíticas que alimentan los modelos. El período abarca múltiples regímenes operativos: la operación regular de días corrientes, los festivos nacionales, las temporadas estacionales (Semana Santa, Carnaval de Barranquilla, Fiestas Patronales) y los ciclos anuales diferenciados.
:::

La fase de análisis exploratorio estableció tres conclusiones que condicionan el diseño de las variables construidas aquí. La demanda presenta estructura multi-escala, con ciclos intra-día bimodales, semanales y anuales superpuestos. El tipo de estacionalidad no actúa como un bloque uniforme y requiere codificación categórica explícita. Y la demanda registrada no equivale a la demanda potencial: una franja sin pasajeros puede reflejar ausencia de despacho, no ausencia de demanda. El modelo predice, por tanto, la demanda efectivamente capturada bajo las condiciones operativas vigentes.

Con esas conclusiones como base, la fase de modelado se formula como un problema de pronóstico supervisado directo multi-horizonte. La demanda se predice por separado para tres ventanas configurables de **30, 60 y 120 minutos**, cada una con su propio modelo. La ventana es rodante desde el instante de decisión e incluye el slot actual, de modo
que la predicción responde a la pregunta operativa concreta: *cuántos pasajeros se esperan en los próximos 30, 60 o 120 minutos a partir de este momento*.

El enfoque es directo, no recursivo. Cada horizonte se predice en un único paso a partir de variables construidas exclusivamente con información disponible en el instante de decisión. No se emplea la predicción de un horizonte como insumo de otro. Esto evita la acumulación de error característica de los esquemas recursivos y respeta la restricción operativa de que el sistema no observa los conteos de pasajeros en tiempo real.

## Columnas de trabajo

El consolidado de entrada contiene nueve columnas, pero no todas entran al conjunto de variables del modelo. La columna `nombre_evento` identifica el evento específico asociado a la franja y su información ya está contenida, de forma categórica y controlada, en `tipo_estacionalidad`. Se excluye para evitar alta cardinalidad y dependencia de etiquetas libres. La columna `n_viajes` registra cuántos viajes fueron despachados en la franja; aunque está disponible en el histórico, tampoco entra al modelo porque contiene información sobre el futuro en el instante de decisión.

Esta distinción es central para el proyecto. Al planear la operación de las próximas horas, el número de buses a despachar es precisamente la decisión que se busca tomar, no un dato conocido. El modelo predice la demanda esperada en la ventana de interés, y el cálculo del número de buses requeridos y del headway correspondiente se realiza en una etapa posterior a partir de esa predicción y de las restricciones operativas de la ruta. Incluir `n_viajes` como variable haría que el modelo aprendiera la oferta en lugar de la demanda, y volvería circular el cálculo del optimizador de frecuencias.

## Variables de calendario

El bloque de calendario aporta la referencia temporal básica de cada
franja. Se construyen tres variables: la hora del día, el día de la
semana y una marca binaria de fin de semana.

La variable `hora` se extrae directamente de la marca temporal asociada
a la franja, que corresponde al inicio del intervalo de 15 minutos.
Representa la hora del día en formato de 24 horas, con valores enteros
entre 0 y 23. Es la referencia que ancla cada observación a su posición
dentro de la jornada operativa y se conserva en formato crudo para
permitir al modelo aprender cortes exactos por franja horaria.

La variable `dia_semana` ya viene incluida en el consolidado y no
requiere transformación. Codifica el día de la semana con valores
enteros de 1 a 7, donde 7 corresponde al domingo. Se conserva tal cual
porque el modelo de árboles puede aprovechar los cortes categóricos
directos sobre ella.

La variable `es_fin_de_semana` se deriva de `dia_semana` como una marca
binaria que toma el valor 1 cuando el día es sábado o domingo, y 0 en
caso contrario:

$$
\text{es\_fin\_de\_semana}_i =
\begin{cases}
1 & \text{si } d_i \in \{6, 7\} \\
0 & \text{en otro caso}
\end{cases}
$$

Su función es exponer de forma explícita la partición más importante
del calendario semanal, ya que el patrón de demanda entre semana y fin
de semana difiere de forma sistemática. Aunque esta información está
implícita en `dia_semana`, presentarla como variable independiente
facilita al modelo identificar la partición sin necesidad de
construirla a partir de múltiples cortes.

## Variables cíclicas

El bloque de calendario sitúa cada franja en su posición exacta dentro del día, la semana y el año, pero no expresa su relación de vecindad con las franjas contiguas. El análisis exploratorio mostró que la serie de demanda tiene estructura periódica marcada: picos semi-diurnos separados por un valle al mediodía, un ciclo semanal con contraste
entre días laborales y fin de semana, y un ciclo anual modulado por temporadas estacionales. Las variables de esta sección codifican esas periodicidades en una forma que los modelos pueden aprovechar directamente. 

:::{admonition} Información: Codificación cíclica
:class: info

Las variables de calendario tienen una naturaleza periódica que una codificación entera no representa adecuadamente. Un modelo lineal o de árboles trata los valores 23 y 0 como lejanos cuando en realidad son contiguos, y lo mismo ocurre con diciembre y enero o con domingo y lunes. Para preservar esa continuidad, cada variable cíclica se descompone en un par seno-coseno sobre su periodo natural:

$$
x_{\sin} = \sin\!\left(\frac{2\pi \cdot x}{P}\right), \qquad
x_{\cos} = \cos\!\left(\frac{2\pi \cdot x}{P}\right)
$$

donde $P$ es el periodo del ciclo. Esta representación proyecta la variable sobre un círculo unitario y garantiza que los extremos del ciclo queden próximos entre sí. Cuando el patrón de la variable no es
sinusoidal simple, se añaden armónicas adicionales que duplican la frecuencia base, permitiendo al modelo representar formas más complejas sin perder la estructura periódica.
:::

Se construyen cuatro niveles de periodicidad (hora, día de la semana, mes y semestre) y en el nivel horario se añade una armónica adicional que responde a la forma bimodal de la demanda intra-día.

### Ciclo horario

La hora del día se codifica sobre un periodo de 24 horas a partir de la hora decimal, que combina `hora` y `minuto` para situar cada franja con precisión de 15 minutos:

$$
h_{\text{dec}} = \text{hora} + \frac{\text{minuto}}{60}
$$

La codificación fundamental captura el ciclo diario completo:

$$
\sin\_hora = \sin\!\left(\frac{2\pi \, h_{\text{dec}}}{24}\right),
\qquad
\cos\_hora = \cos\!\left(\frac{2\pi \, h_{\text{dec}}}{24}\right)
$$

Una sinusoide de un solo periodo solo puede representar un pico por día, pero la demanda observada presenta **dos picos** (mañana y tarde) separados por un valle al mediodía. Para capturar esa forma bimodal se añade una segunda armónica que duplica la frecuencia base:

$$
\sin\_hora\_2 = \sin\!\left(\frac{4\pi \, h_{\text{dec}}}{24}\right),
\qquad
\cos\_hora\_2 = \cos\!\left(\frac{4\pi \, h_{\text{dec}}}{24}\right)
$$

El par fundamental y su segunda armónica, en conjunto, permiten al modelo reconstruir la doble joroba sin necesidad de tratar cada franja como una categoría independiente.

### Ciclo semanal

El día de la semana se codifica sobre un periodo de 7 días:

$$
\sin\_dow = \sin\!\left(\frac{2\pi \, \text{dia\_semana}}{7}\right),
\qquad
\cos\_dow = \cos\!\left(\frac{2\pi \, \text{dia\_semana}}{7}\right)
$$

Esta representación captura la continuidad entre domingo y lunes que la variable entera `dia_semana` no refleja, y complementa la marca binaria `es_fin_de_semana` sin reemplazarla.

### Ciclo anual

El mes del año se codifica sobre un periodo de 12 meses:

$$
\sin\_mes = \sin\!\left(\frac{2\pi \, \text{mes}}{12}\right),
\qquad
\cos\_mes = \cos\!\left(\frac{2\pi \, \text{mes}}{12}\right)
$$

La codificación preserva la continuidad entre diciembre y enero y permite al modelo representar el ciclo anual sin depender del orden consecutivo de los meses.

### Armónica semestral

Al igual que en el ciclo horario, el ciclo anual se refuerza con una segunda armónica que duplica su frecuencia, sobre un periodo de 6 meses:

$$
\sin\_semestre = \sin\!\left(\frac{2\pi \, \text{mes}}{6}\right),
\qquad
\cos\_semestre = \cos\!\left(\frac{2\pi \, \text{mes}}{6}\right)
$$

Esta armónica permite al modelo representar formas anuales con más de un máximo o con asimetrías entre la primera y la segunda mitad del año. Su aporte real se valida en la etapa de selección de variables.

## Variables de estacionalidad

El análisis exploratorio mostró que la demanda de pasajeros cambia de forma sistemática según el régimen estacional del día. Los días corrientes y las fiestas de Atlántico sostienen la demanda en niveles altos, mientras que la Temporada Nacional y los Festivos Nacionales la reducen con magnitudes distintas. La categoría es nominal y de baja
cardinalidad (cinco valores posibles), y cada día pertenece a una y solo una de ellas, por lo que la codificación natural es **one-hot**.

:::{admonition} Información: Categorías de estacionalidad
:class: info

La clasificación estacional utilizada no es propia del equipo, sino que proviene de la categorización oficial definida por SOBUSA y Extreme Technologies en el consolidado operacional. Se respeta tal cual para mantener consistencia con la operación real y con la nomenclatura utilizada por el negocio.

| Categoría | Temporadas que agrupa |
|---|---|
| `DIAS_CORRIENTES` | Días corrientes del año (temporada base). |
| `FESTIVO_NACIONAL` | Festivos nacionales, por ejemplo Año Nuevo (1 de enero). |
| `FIESTA_ATLANTICO` | Precarnaval y Carnaval de Barranquilla, por ejemplo el sábado de Carnaval. |
| `TEMPORADA_NACIONAL` | Semana Santa, por ejemplo el Viernes Santo. |
| `FIESTAS_PATRONALES` | Fiestas de la Virgen del Carmen, por ejemplo el 16 de julio. |

Esta clasificación se conserva en el conjunto de variables sin reagrupamientos ni fusiones entre categorías, de modo que el modelo pueda aprender el efecto de cada régimen por separado y las recomendaciones posteriores puedan leerse con la misma granularidad con la que opera el negocio.:::

La codificación genera cinco columnas binarias, una por categoría:

$$
\text{est\_}c_i = \mathbb{1}\left[\, \text{tipo\_estacionalidad}_i = c \,\right],
\qquad c \in \{\text{DIAS\_CORRIENTES}, \dots, \text{FIESTAS\_PATRONALES}\}
$$

Como las cinco categorías son mutuamente excluyentes, en cada fila exactamente una columna vale 1 y las demás valen 0. Esta propiedad es la que permite al modelo inferir el régimen estacional sin ambigüedad y sin recurrir a una codificación ordinal, que introduciría un orden artificial entre categorías sin relación de magnitud.

En un modelo lineal, la suma constante de las cinco columnas generaría colinealidad perfecta con el intercepto y obligaría a eliminar una categoría como base, dejando cuatro binarias. En los modelos de árboles utilizados en este proyecto la redundancia no produce problemas numéricos, y presentar las cinco categorías por separado facilita la lectura de la importancia de variables y la construcción de explicaciones locales.

## Variables de rezago

Las variables de rezago son el componente central de este bloque de ingeniería. Son las que inyectan la **temporalidad** al modelo: sin ellas, cada franja se modelaría como una observación independiente y el histórico reciente no tendría ningún peso en la predicción. La demanda de una franja no depende solo de su posición en el calendario, sino también de lo que ocurrió en franjas previas, y ese vínculo se materializa a través de los rezagos.

Cada variable `lag_k` se construye como el valor de `pasajeros` en la franja que se encuentra `k` posiciones antes de la franja actual dentro de la serie ordenada cronológicamente:

$$
\text{lag\_k}_i = \text{pasajeros}_{i-k}
$$

La elección de los valores de `k` no es arbitraria. Proviene de la estructura de autocorrelación identificada en el análisis exploratorio, que mostró picos semi-diurnos en los rezagos 32–33 y un ciclo diario completo en el rezago 65. La tabla siguiente resume los rezagos utilizados, su interpretación temporal sobre franjas de 15 minutos y la escala que cada uno captura:

| Rezago | Desplazamiento | Escala que captura |
|---|---|---|
| `lag_32` | 8 horas | Ciclo semi-diurno (pico opuesto del mismo día) |
| `lag_33` | 8 horas 15 min | Ciclo semi-diurno, un slot desplazado |
| `lag_65` | 1 día | Mismo slot del día anterior |
| `lag_455` | 7 días | Mismo slot de la semana anterior |

La combinación de estos cuatro rezagos cubre dos escalas temporales relevantes. Los pares `lag_32` y `lag_33` capturan la forma bimodal intra-día: la demanda de una franja se relaciona con la demanda de la
franja situada aproximadamente ocho horas antes, que corresponde al pico opuesto de la jornada. Los rezagos `lag_65` y `lag_455` capturan la persistencia a nivel diario y semanal, que en la práctica es el
patrón más fuerte de la serie: la mejor referencia para estimar la demanda de un lunes a las 10:00 es la demanda del lunes anterior a las 10:00, y en segunda instancia la del lunes de la semana previa.

:::{admonition} Información: ¿Por qué no se usan rezagos más cortos?
:class: info

El análisis exploratorio identificó además estructura de corto plazo en los rezagos 1 y 4, asociada a memoria inmediata (15 minutos) y horaria (1 hora). Esos rezagos **no se incorporan** al conjunto de variables por una restricción operativa concreta.
 
 En el instante de decisión, el sistema no conoce los conteos de pasajeros de las franjas inmediatamente anteriores. Un viaje que despacha a las 10:00 puede tardar entre una y varias horas en cerrar su recorrido, y el dispositivo de conteo sincroniza con retraso adicional. Con una duración mediana de recorrido de 200 minutos y un retraso de sincronización estimado, el conteo de una franja no está disponible hasta aproximadamente 5 horas después de su cierre. Esto descarta cualquier rezago por debajo de ~20 posiciones.
 
Los rezagos de 32, 33, 65 y 455 quedan muy por encima de ese umbral y por lo tanto son **operativamente seguros**: representan información que, en el momento de predecir, ya está consolidada y disponible.
:::

Los rezagos heredan el `NaN` natural de sus bordes: las primeras 32 posiciones de la serie no tienen valor para `lag_32`, las primeras 65 no lo tienen para `lag_65`, y las primeras 455 no lo tienen para `lag_455`. 

## Implementación

La lógica descrita en las secciones anteriores se encapsula en la función `build_features`, definida en el módulo `features/build_features.py` del repositorio. La función recibe un split (entrenamiento o validación) y devuelve el mismo conjunto enriquecido con las variables derivadas, respetando el orden cronológico original y sin transformar la columna de respuesta. La definición y la implementación concreta de cada variable se resumen en la tabla siguiente.

| Bloque          | Feature                | Tipo          | Deriva de            | ¿Cómo se deriva?                    |
|-----------------|------------------------|---------------|----------------------|-----------------------------------|
| Calendario      | hora                   | int           | franja (timestamp)   | franja.dt.hour                    |
| Calendario      | dia_semana             | int           | df_agre              | ya viene del consolidado          |
| Calendario      | es_fin_de_semana       | binaria       | dia_semana           | dia_semana.isin([6, 7])           |
| Cíclico         | sin_hora               | float         | hora, minuto         | sin(2π · hora_decimal / 24)       |
| Cíclico         | cos_hora               | float         | hora, minuto         | cos(2π · hora_decimal / 24)       |
| Cíclico         | sin_hora_2             | float         | hora, minuto         | sin(4π · hora_decimal / 24)       |
| Cíclico         | cos_hora_2             | float         | hora, minuto         | cos(4π · hora_decimal / 24)       |
| Cíclico         | sin_dow                | float         | dia_semana           | sin(2π · dow / 7)                 |
| Cíclico         | cos_dow                | float         | dia_semana           | cos(2π · dow / 7)                 |
| Cíclico         | sin_mes                | float         | mes                  | sin(2π · mes / 12)                |
| Cíclico         | cos_mes                | float         | mes                  | cos(2π · mes / 12)                |
| Cíclico         | sin_semestre           | float         | mes                  | sin(2π · mes / 6)                 |
| Cíclico         | cos_semestre           | float         | mes                  | cos(2π · mes / 6)                 |
| Estacionalidad  | est_DIAS_CORRIENTES    | binaria       | tipo_estacionalidad  | one-hot                           |
| Estacionalidad  | est_FESTIVO_NACIONAL   | binaria       | tipo_estacionalidad  | one-hot                           |
| Estacionalidad  | est_FIESTA_ATLANTICO   | binaria       | tipo_estacionalidad  | one-hot                           |
| Estacionalidad  | est_TEMPORADA_NACIONAL | binaria       | tipo_estacionalidad  | one-hot                           |
| Estacionalidad  | est_FIESTAS_PATRONALES | binaria       | tipo_estacionalidad  | one-hot                           |
| Lags            | lag_32                 | float         | pasajeros            | pasajeros.shift(32)               |
| Lags            | lag_33                 | float         | pasajeros            | pasajeros.shift(33)               |
| Lags            | lag_65                 | float         | pasajeros            | pasajeros.shift(65)               |
| Lags            | lag_455                | float         | pasajeros            | pasajeros.shift(455)              |

El conjunto se diseñó de manera que la ingesta por parte de los entrenadores de los modelos sea directa y homogénea. Todas las variables tienen tipo numérico, sin valores categóricos en texto ni estructuras anidadas, y el esquema de columnas es idéntico entre los segmentos de entrenamiento y de validación. Esa consistencia permite avanzar sobre ambos segmentos de forma fiable: cada modelo recibe exactamente las mismas columnas en el mismo orden, y las diferencias entre splits se reducen a los valores de las filas, no a su estructura.

## Pruebas

Se realizaron dos pruebas independientes sobre los datasets derivados de la ruta K54, una sobre el segmento de entrenamiento y otra sobre el de validación. En ambos casos, el pipeline de ingeniería de variables se aplicó con `dropna=False` en una primera pasada para inspeccionar el resultado completo, y con `dropna=True` en una segunda pasada para obtener el conjunto listo para el modelado.

El descarte elimina las primeras 455 franjas del bloque, que equivalen a los primeros siete días de la serie. Esas franjas iniciales se usan para construir los rezagos de las filas posteriores y, una vez cumplen ese papel, se retiran del conjunto modelable: el rezago más largo, `lag_455`, requiere 455 franjas de historia previa para calcularse, y las primeras filas no disponen de ella. No es un problema de datos faltantes, sino de haber agotado su función como contexto.

In [15]:
train_feat = bf.build_features(train, dropna=True)
_ = toolkit.dataframe_head_table(train_feat, n=5)

franja,hora,dia_semana,es_fin_de_semana,sin_hora,cos_hora,sin_hora_2,cos_hora_2,sin_dow,cos_dow,sin_mes,cos_mes,sin_semestre,cos_semestre,est_DIAS_CORRIENTES,est_FESTIVO_NACIONAL,est_FIESTAS_PATRONALES,est_FIESTA_ATLANTICO,est_TEMPORADA_NACIONAL,lag_32,lag_33,lag_65,lag_455,pasajeros
04:30:00,4,7,1,0.923880,0.382683,0.707107,-0.707107,-0.000000,1.000000,0.500000,0.866025,0.866025,0.500000,1,0,0,0,0,458,307,167,0,0
04:45:00,4,7,1,0.946930,0.321439,0.608761,-0.793353,-0.000000,1.000000,0.500000,0.866025,0.866025,0.500000,1,0,0,0,0,310,458,181,0,0
05:00:00,5,7,1,0.965926,0.258819,0.500000,-0.866025,-0.000000,1.000000,0.500000,0.866025,0.866025,0.500000,1,0,0,0,0,375,310,419,220,283
05:15:00,5,7,1,0.980785,0.195090,0.382683,-0.923880,-0.000000,1.000000,0.500000,0.866025,0.866025,0.500000,1,0,0,0,0,240,375,369,116,166
05:30:00,5,7,1,0.991445,0.130526,0.258819,-0.965926,-0.000000,1.000000,0.500000,0.866025,0.866025,0.500000,1,0,0,0,0,343,240,384,56,209


El bloque de validación reproduce el mismo comportamiento. El conjunto arranca en el corte cronológico establecido por `split_and_save` y la segunda pasada descarta las 455 franjas iniciales correspondientes a los primeros siete días posteriores al corte. En la salida se observan las mismas características que en el bloque de entrenamiento: los cuatro rezagos aparecen poblados con valores enteros, las variables de calendario, codificación cíclica y estacionalidad están completas, y `pasajeros` conserva su valor original en cada franja.

In [14]:
holdout_feat = bf.build_features(holdout, dropna=True)
_ = toolkit.dataframe_head_table(holdout_feat, n=5)

franja,hora,dia_semana,es_fin_de_semana,sin_hora,cos_hora,sin_hora_2,cos_hora_2,sin_dow,cos_dow,sin_mes,cos_mes,sin_semestre,cos_semestre,est_DIAS_CORRIENTES,est_FESTIVO_NACIONAL,est_FIESTAS_PATRONALES,est_FIESTA_ATLANTICO,est_TEMPORADA_NACIONAL,lag_32,lag_33,lag_65,lag_455,pasajeros
17:30:00,17,1,0,-0.991445,-0.130526,0.258819,-0.965926,0.781831,0.623490,0.000000,-1.000000,-0.000000,1.000000,0,1,0,0,0,183,168,183,181,242
17:45:00,17,1,0,-0.997859,-0.065403,0.130526,-0.991445,0.781831,0.623490,0.000000,-1.000000,-0.000000,1.000000,0,1,0,0,0,114,183,101,182,156
18:00:00,18,1,0,-1.000000,-0.000000,0.000000,-1.000000,0.781831,0.623490,0.000000,-1.000000,-0.000000,1.000000,0,1,0,0,0,162,114,204,228,180
18:15:00,18,1,0,-0.997859,0.065403,-0.130526,-0.991445,0.781831,0.623490,0.000000,-1.000000,-0.000000,1.000000,0,1,0,0,0,184,162,122,253,157
18:30:00,18,1,0,-0.991445,0.130526,-0.258819,-0.965926,0.781831,0.623490,0.000000,-1.000000,-0.000000,1.000000,0,1,0,0,0,159,184,196,131,261


## Conclusiones

La libreta concluye con un pipeline de ingeniería de características que entrega un conjunto enriquecido con las 22 variables predictoras y la columna de respuesta `pasajeros` intacta, conservando el DatetimeIndex completo (`franja_ts`) para permitir la construcción de los horizontes de predicción. El esquema de columnas es idéntico entre los bloques de entrenamiento y de validación, sin valores nulos en las variables predictoras y con los tipos de dato consistentes entre ambos. El pipeline mantiene el mismo contrato de entrada para los dos conjuntos, de modo que cualquier split futuro se procesa con la misma función y produce la misma estructura de salida.

Existen fuentes adicionales de señal temporal que no se incorporaron en esta versión y quedan documentadas como alternativas para una iteración posterior. Entre ellas, los promedios móviles sobre los `k` rezagos seguros más recientes, las medias y desviaciones del mismo slot sobre los últimos siete días, y los agregados horarios del día anterior. Estas variables suavizan el ruido de un único rezago y capturan el nivel y la volatilidad del comportamiento reciente, pero introducen superficie adicional en el modelo. Se decidió comenzar con el conjunto actual (el que reproduce la estructura multi-escala identificada en el análisis exploratorio) y evaluar su desempeño antes de incorporar agregados, para poder atribuir con claridad cualquier mejora o degradación a una causa concreta.

La construcción de los targets (30, 60 y 120 minutos) no forma parte de esta libreta y corresponde a la fase de modelado. La respuesta `pasajeros` se entrega sin transformaciones, y cada modelo define cómo agregarla sobre la ventana rodante desde el instante de decisión.